## 1. Setup Repository and Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/Adr44mo/ML4VM_yolov8n.git
%cd ML4VM_yolov8n

In [ ]:
# Install required packages
!pip install -q ultralytics opencv-python pyyaml matplotlib numpy torch torchvision

In [ ]:
# Import necessary libraries
import os
import sys
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Add utils to path
sys.path.append('utils')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download and Setup Datasets

### Option A: Download COCO val2017 (small subset)
This will download a small subset of COCO validation images (~1GB)

In [ ]:
# Create COCO directory structure
!mkdir -p COCO/images/val2017
!mkdir -p COCO/labels/val2017

# Download COCO val2017 images (this takes a while - ~1GB)
!wget -q http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip -d COCO/images/
!rm val2017.zip

# Download COCO annotations
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip -d COCO/
!rm annotations_trainval2017.zip

print("COCO dataset downloaded!")

### Option B: Mount Google Drive (if you have COCO already)
Uncomment and modify paths if you have COCO on Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Create symlink to your COCO dataset on Drive
# !ln -s /content/drive/MyDrive/datasets/COCO ./COCO

### Download OVAD Dataset

In [ ]:
# Create ovad directory
!mkdir -p ovad

# Download OVAD annotations (adjust URL if needed)
# You may need to download from the official OVAD source
# For now, we'll assume you have it or can upload it
print("Please upload ovad2000.json to the ovad/ directory")
print("You can download it from: https://prior.allenai.org/projects/ovad")

# Or upload from local:
# from google.colab import files
# uploaded = files.upload()
# !mv ovad2000.json ovad/

In [ ]:
# Generate YOLO format labels from COCO
!python convert_coco_to_yolo.py

# Generate val2017.txt file list
!python generate_val_txt.py

print("Dataset setup complete!")

## 3. Import Model Components

Import all the YOLOv8 components from the repository

In [ ]:
# Import model components from the repository
import torch.nn as nn

# Import YOLOv8 building blocks
exec(open('model_components.py').read())

# Import pose detection components
from head_with_pose import HeadWithPose
from ovad_dataset import OVADDataset
from loss_with_pose import ComputeLossWithPose

print("Model components imported successfully!")

## 4. Load Configuration and Dataset

In [ ]:
# Load parameters
with open('utils/args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

# Check if OVAD file exists
ovad_json_path = 'ovad/ovad2000.json'
coco_img_dir = 'COCO/images/val2017'

if not os.path.exists(ovad_json_path):
    print(f"⚠️ Warning: OVAD file not found at {ovad_json_path}")
    print("Please upload ovad2000.json to the ovad/ directory")
else:
    # Create OVAD dataset
    ovad_dataset = OVADDataset(
        ovad_json_path=ovad_json_path,
        coco_img_dir=coco_img_dir,
        input_size=640,
        params=params,
        augment=True  # Enable augmentation for training
    )
    
    print(f"✓ Dataset loaded successfully!")
    print(f"  Number of images: {len(ovad_dataset)}")
    print(f"  Pose classes: {ovad_dataset.pose_classes}")
    
    # Create dataloader
    batch_size = 8  # Adjust based on GPU memory
    pose_loader = DataLoader(
        ovad_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=OVADDataset.collate_fn
    )
    
    print(f"  Batch size: {batch_size}")
    print(f"  Number of batches: {len(pose_loader)}")

## 5. Initialize Model

In [ ]:
# Create model with pose detection head
class MyYoloWithPose(nn.Module):
    def __init__(self, version):
        super().__init__()
        self.backbone = Backbone(version=version)
        self.neck = Neck(version=version)
        self.head = HeadWithPose(version=version, num_classes=80, num_poses=3)

    def forward(self, x):
        x = self.backbone(x)
        x = self.neck(x[0], x[1], x[2])
        return self.head(list(x))

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_pose = MyYoloWithPose(version='n').to(device)
model_pose.head.stride = torch.tensor([8.0, 16.0, 32.0])

total_params = sum(p.numel() for p in model_pose.parameters())
print(f"\nModel initialized: {total_params/1e6:.2f}M parameters")
print(f"  - Backbone + Neck: {sum(p.numel() for p in model_pose.backbone.parameters())/1e6:.2f}M + {sum(p.numel() for p in model_pose.neck.parameters())/1e6:.2f}M")
print(f"  - Head (with pose): {sum(p.numel() for p in model_pose.head.parameters())/1e6:.2f}M")

## 6. Setup Training

In [ ]:
# Setup loss and optimizer
params_pose = params.copy()
params_pose['pose'] = 1.0  # weight for pose loss

criterion_pose = ComputeLossWithPose(model_pose, params_pose, num_poses=3)
optimizer_pose = torch.optim.AdamW(model_pose.parameters(), lr=0.001, weight_decay=0.0005)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pose, T_max=50, eta_min=1e-5)

print("Training setup complete!")
print(f"  Optimizer: AdamW (lr=0.001)")
print(f"  Loss weights: box={params_pose.get('box', 7.5)}, cls={params_pose.get('cls', 0.5)}, dfl={params_pose.get('dfl', 1.5)}, pose={params_pose['pose']}")

## 7. Training Loop

In [ ]:
# Training configuration
num_epochs = 50
save_interval = 5  # Save checkpoint every N epochs
log_interval = 10  # Log every N batches

# Create checkpoints directory
os.makedirs('checkpoints', exist_ok=True)

# Training history
history = {
    'epoch': [],
    'total_loss': [],
    'box_loss': [],
    'cls_loss': [],
    'dfl_loss': [],
    'pose_loss': []
}

print(f"Starting training for {num_epochs} epochs...\n")
print("=" * 80)

model_pose.train()
for epoch in range(num_epochs):
    epoch_losses = {'total': 0, 'box': 0, 'cls': 0, 'dfl': 0, 'pose': 0}
    
    for batch_idx, (images, targets) in enumerate(pose_loader):
        # Move to device
        images = images.float().to(device)
        targets['cls'] = targets['cls'].to(device)
        targets['box'] = targets['box'].to(device)
        targets['idx'] = targets['idx'].to(device)
        targets['pose'] = targets['pose'].to(device)
        
        # Forward pass
        outputs = model_pose(images)
        
        # Compute losses
        loss_box, loss_cls, loss_dfl, loss_pose = criterion_pose(outputs, targets)
        total_loss = loss_box + loss_cls + loss_dfl + loss_pose
        
        # Backward pass
        optimizer_pose.zero_grad()
        total_loss.backward()
        optimizer_pose.step()
        
        # Accumulate losses
        epoch_losses['total'] += total_loss.item()
        epoch_losses['box'] += loss_box.item()
        epoch_losses['cls'] += loss_cls.item()
        epoch_losses['dfl'] += loss_dfl.item()
        epoch_losses['pose'] += loss_pose.item()
        
        # Log progress
        if (batch_idx + 1) % log_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx+1}/{len(pose_loader)}] - "
                  f"Loss: {total_loss.item():.4f} (box: {loss_box.item():.4f}, "
                  f"cls: {loss_cls.item():.4f}, dfl: {loss_dfl.item():.4f}, pose: {loss_pose.item():.4f})")
    
    # Update learning rate
    scheduler.step()
    
    # Calculate average losses
    num_batches = len(pose_loader)
    avg_total = epoch_losses['total'] / num_batches
    avg_box = epoch_losses['box'] / num_batches
    avg_cls = epoch_losses['cls'] / num_batches
    avg_dfl = epoch_losses['dfl'] / num_batches
    avg_pose = epoch_losses['pose'] / num_batches
    
    # Save to history
    history['epoch'].append(epoch + 1)
    history['total_loss'].append(avg_total)
    history['box_loss'].append(avg_box)
    history['cls_loss'].append(avg_cls)
    history['dfl_loss'].append(avg_dfl)
    history['pose_loss'].append(avg_pose)
    
    # Print epoch summary
    print(f"\n{'='*80}")
    print(f"Epoch [{epoch+1}/{num_epochs}] Summary:")
    print(f"  Average Total Loss: {avg_total:.4f}")
    print(f"  Box Loss: {avg_box:.4f} | Cls Loss: {avg_cls:.4f} | DFL Loss: {avg_dfl:.4f} | Pose Loss: {avg_pose:.4f}")
    print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
    print(f"{'='*80}\n")
    
    # Save checkpoint
    if (epoch + 1) % save_interval == 0 or (epoch + 1) == num_epochs:
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model_pose.state_dict(),
            'optimizer_state_dict': optimizer_pose.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'params': params_pose
        }
        checkpoint_path = f'checkpoints/yolov8n_pose_epoch_{epoch+1}.pt'
        torch.save(checkpoint, checkpoint_path)
        print(f"✓ Checkpoint saved: {checkpoint_path}\n")

print("\n" + "="*80)
print("Training complete!")
print(f"Final model saved at: checkpoints/yolov8n_pose_epoch_{num_epochs}.pt")
print("="*80)

## 8. Plot Training History

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Total loss
axes[0, 0].plot(history['epoch'], history['total_loss'], 'b-', linewidth=2)
axes[0, 0].set_title('Total Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# Box loss
axes[0, 1].plot(history['epoch'], history['box_loss'], 'r-', linewidth=2)
axes[0, 1].set_title('Box Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].grid(True, alpha=0.3)

# Class loss
axes[0, 2].plot(history['epoch'], history['cls_loss'], 'g-', linewidth=2)
axes[0, 2].set_title('Classification Loss', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].grid(True, alpha=0.3)

# DFL loss
axes[1, 0].plot(history['epoch'], history['dfl_loss'], 'm-', linewidth=2)
axes[1, 0].set_title('DFL Loss', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True, alpha=0.3)

# Pose loss
axes[1, 1].plot(history['epoch'], history['pose_loss'], 'c-', linewidth=2)
axes[1, 1].set_title('Pose Loss', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True, alpha=0.3)

# All losses combined
axes[1, 2].plot(history['epoch'], history['box_loss'], 'r-', label='Box', linewidth=1.5, alpha=0.7)
axes[1, 2].plot(history['epoch'], history['cls_loss'], 'g-', label='Cls', linewidth=1.5, alpha=0.7)
axes[1, 2].plot(history['epoch'], history['dfl_loss'], 'm-', label='DFL', linewidth=1.5, alpha=0.7)
axes[1, 2].plot(history['epoch'], history['pose_loss'], 'c-', label='Pose', linewidth=1.5, alpha=0.7)
axes[1, 2].set_title('All Losses', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Loss')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training curves saved to: checkpoints/training_curves.png")

## 9. Test Inference on Sample Images

In [ ]:
import matplotlib.patches as patches

# Load the trained model
model_pose.eval()

# Get a test image
image_idx = 0  # Change to test different images
test_img_tensor, test_labels = ovad_dataset[image_idx]

# Convert to display format
test_img_display = test_img_tensor.numpy().transpose(1, 2, 0).astype(np.uint8)

# Run inference
test_img_batch = test_img_tensor.unsqueeze(0).float().to(device)
with torch.no_grad():
    predictions = model_pose(test_img_batch)

# Extract predictions
pred_boxes = predictions[0, :4, :].cpu()
pred_class_scores = predictions[0, 4:84, :].cpu()
pred_pose_scores = predictions[0, 84:87, :].cpu()

# Filter for high-confidence person detections
person_class_id = 0
confidence_threshold = 0.25
person_scores = pred_class_scores[person_class_id, :]
high_conf_mask = person_scores > confidence_threshold

# Get filtered predictions
filtered_boxes = pred_boxes[:, high_conf_mask]
filtered_poses = pred_pose_scores[:, high_conf_mask].argmax(dim=0)
filtered_scores = person_scores[high_conf_mask]

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

pose_names = ['lying', 'sitting', 'standing']
pose_colors = ['red', 'blue', 'green']

# Ground truth
ax1.imshow(test_img_display)
ax1.set_title('Ground Truth', fontsize=14)
ax1.axis('off')

if len(test_labels) > 0:
    for i in range(test_labels.shape[0]):
        x_c, y_c, w, h = test_labels[i, 1:5].numpy()
        pose_id = int(test_labels[i, 5].item())
        
        x_c *= 640
        y_c *= 640
        w *= 640
        h *= 640
        
        x1 = x_c - w/2
        y1 = y_c - h/2
        
        rect = patches.Rectangle((x1, y1), w, h, linewidth=2, 
                                edgecolor=pose_colors[pose_id], facecolor='none')
        ax1.add_patch(rect)
        ax1.text(x1, y1-5, f'GT: {pose_names[pose_id]}', 
                color=pose_colors[pose_id], fontsize=10, 
                bbox=dict(facecolor='white', alpha=0.7))

# Predictions
ax2.imshow(test_img_display)
ax2.set_title(f'Predictions (conf > {confidence_threshold})', fontsize=14)
ax2.axis('off')

for i in range(filtered_boxes.shape[1]):
    x_c, y_c, w, h = filtered_boxes[:, i].numpy()
    pose_id = int(filtered_poses[i].item())
    score = filtered_scores[i].item()
    
    x1 = x_c - w/2
    y1 = y_c - h/2
    
    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, 
                            edgecolor=pose_colors[pose_id], facecolor='none')
    ax2.add_patch(rect)
    ax2.text(x1, y1-5, f'{pose_names[pose_id]}: {score:.2f}', 
            color=pose_colors[pose_id], fontsize=10, 
            bbox=dict(facecolor='white', alpha=0.7))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='none', edgecolor=pose_colors[i], 
                        label=pose_names[i], linewidth=2) for i in range(3)]
ax2.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('checkpoints/inference_example.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Ground truth boxes: {test_labels.shape[0]}")
print(f"Predicted boxes: {filtered_boxes.shape[1]}")

## 10. Download Trained Model

Download the trained model and training curves to your local machine

In [ ]:
# Zip all checkpoints and results
!zip -r trained_model.zip checkpoints/

# Download to local machine
from google.colab import files
files.download('trained_model.zip')

print("Model and results ready for download!")